In [ ]:
from cobra.io import read_sbml_model

model = read_sbml_model("../model/Rpom_05.xml")

In [ ]:
from collections import Counter


def get_elemental_composition(rpom_model):
    # List of macromolecular pseudo-metabolites
    pseudo_mets = {
        "PROTEIN[c]": "PROTEIN-RXN",
        "DNA[c]": "DNA-RXN",
        "RNA[c]": "RNA-RXN",
        "LIPID[c]": "LIPID-RXN",
        "COFACTORS[c]": "COFACTOR-RXN",
        "IONS[c]": "IONS-RXN",
        "MUREIN[c]": "MUREIN-RXN",
        "PHB-STORAGE[c]": "PHB-STORAGE-RXN",
    }

    # For each pseudo-metabolite, add up element counts of its components
    # weighted by their mmol/g contribution to total
    elemental_stoich = {}
    for pseudo_met, rxn in pseudo_mets.items():
        pseudo_met_stoich = Counter()
        composition = rpom_model.reactions.get_by_id(
            rxn
        ).metabolites  # in mmol component per g pseudo-metabolite

        # mmol of each element per g pseudo-metabolite
        for met, coeff in composition.items():
            weighted_elements = {
                elem: -count * coeff for elem, count in met.elements.items()
            }
            pseudo_met_stoich.update(weighted_elements)

        elemental_stoich[pseudo_met] = pseudo_met_stoich

    # Add up over full biomass reaction
    result = Counter()
    for met, coeff in rpom_model.reactions.get_by_id(
        "Rpom_hwa_biomass"
    ).metabolites.items():
        if met.id in elemental_stoich:
            # Use values calculated above, scaled by biomass proportion
            result.update(
                {
                    elem: -amount * coeff
                    for elem, amount in elemental_stoich[met.id].items()
                }
            )
        else:
            # Not one of the proportionally scaled pseudo-metabolites,
            # use energy counts multiplied by mmol / gDCW instead
            result.update(
                {
                    elem: -count * coeff
                    for elem, count in met.elements.items()
                }
            )

    return result


get_elemental_composition(model)

Counter({'H': 58.246996699085464,
         'C': 33.1483819153176,
         'O': 14.203626277567793,
         'N': 7.420365027341404,
         'P': 0.44543357243631476,
         'S': 0.20916695662632612,
         'K': 0.1951930019508361,
         'Fe': 0.014746000136326041,
         'Mg': 0.008675000087107686,
         'Ca': 0.0052050000483346084,
         'Cl': 0.0052050000483346084,
         'Cu': 0.000709000002265056,
         'Mn': 0.000691000006101525,
         'Zn': 0.00034100000646613204,
         'Ni': 0.0003230000004775921,
         'Co': 2.5000000675743e-05,
         'Mo': 7.000004512212e-06})